# Build External Data Pipeline

Use this notebook to test the first five external-data steps interactively before turning them into production ingestion jobs. The notebook is free-first and designed around official filings, macro data, broad news discovery, and optional market-data enrichment.


## 0. Setup

- `SEC` = **U.S. Securities and Exchange Commission**. We use SEC data for company master data, filing metadata, company facts, and filing text.
- `FRED` = **Federal Reserve Economic Data** from the Federal Reserve Bank of St. Louis. We use it for macro series like rates, inflation, and unemployment.
- `GDELT` = **Global Database of Events, Language, and Tone**. We use it for broad news and narrative discovery.
- `FMP` = **Financial Modeling Prep**. It is optional and only runs if `FMP_API_KEY` is set in the environment.

### Helpful Short Forms

- `CIK` = **Central Index Key**, the SEC company identifier.
- `API` = **Application Programming Interface**, which is how we request data programmatically.
- `10-K` = annual filing.
- `10-Q` = quarterly filing.
- `8-K` = current event filing for important company updates.
- `MD&A` = **Management’s Discussion and Analysis**, where management explains company performance and risks.
- `Vector DB` = **Vector Database**, used for storing searchable text embeddings rather than numeric market tables.


In [54]:
from __future__ import annotations
import gzip
import json
import os
import re
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request, urlopen
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
RAW_DIR = DATA_DIR / 'raw'
load_dotenv(ROOT / '.env')
for path in [EXTERNAL_DIR / 'sec', EXTERNAL_DIR / 'fred', EXTERNAL_DIR / 'gdelt', EXTERNAL_DIR / 'fmp', INTERIM_DIR / 'vector_preview']:
    path.mkdir(parents=True, exist_ok=True)

SEC_HEADERS = {
    'User-Agent': 'portfolio-analyzer research contact@example.com',
    'Accept-Encoding': 'gzip, deflate',
}

def read_response_bytes(response) -> bytes:
    payload = response.read()
    encoding = (response.headers.get('Content-Encoding') or '').lower()
    if 'gzip' in encoding:
        payload = gzip.decompress(payload)
    return payload

def fetch_url(url: str, headers: dict[str, str] | None = None, *, retries: int = 4, base_delay: float = 1.5) -> bytes:
    merged_headers = headers or {}
    last_error = None
    for attempt in range(retries):
        request = Request(url, headers=merged_headers)
        try:
            with urlopen(request) as response:
                return read_response_bytes(response)
        except HTTPError as error:
            last_error = error
            if error.code != 429 or attempt == retries - 1:
                raise
            retry_after = error.headers.get("Retry-After")
            delay = float(retry_after) if retry_after else base_delay * (2 ** attempt)
            print(f"Rate limited for {url}. Sleeping {delay:.1f}s before retry {attempt + 2}/{retries}.")
            time.sleep(delay)
    raise last_error if last_error else RuntimeError("Request failed without a captured HTTPError")

def fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:
    return json.loads(fetch_url(url, headers=headers).decode('utf-8'))

def fetch_text(url: str, headers: dict[str, str] | None = None) -> str:
    return fetch_url(url, headers=headers).decode('utf-8', errors='ignore')

def simple_chunk(text: str, chunk_size: int = 1400) -> list[str]:
    clean = re.sub(r'\s+', ' ', text).strip()
    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]

def html_to_text(html: str) -> str:
    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)
    no_tags = re.sub(r'<[^>]+>', ' ', no_script)
    return re.sub(r'\s+', ' ', no_tags).strip()

fake_csv = RAW_DIR / 'fake_mantis_invest.csv'
transactions = pd.read_csv(fake_csv) if fake_csv.exists() else pd.DataFrame()
tickers = sorted({str(t).strip() for t in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist() if str(t).strip()})
tickers[:10], len(tickers)

(['AAPL',
  'AMZN',
  'AVGO',
  'BRK.B',
  'GOOGL',
  'META',
  'MSFT',
  'NVDA',
  'QQQ',
  'TSLA'],
 12)

## 1. Master Data

Build the ticker universe, map it to SEC CIKs, and create the company master table. This becomes the base join key for the rest of the pipeline.


In [55]:
company_master.head()

,cik,ticker,company_name
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC


In [62]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'
company_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)
company_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})
company_master['ticker'] = company_master['ticker'].astype(str).str.upper()
company_master['cik'] = company_master['cik'].astype(str).str.zfill(10)
if tickers:
    company_master = company_master[company_master['ticker'].isin(tickers)].copy()
company_master.to_csv(EXTERNAL_DIR / 'sec' / 'company_master.csv', index=False)
company_master.head(10)

,cik,ticker,company_name
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
5,0001730168,AVGO,Broadcom Inc.
6,0001326801,META,"Meta Platforms, Inc."
7,0001318605,TSLA,"Tesla, Inc."
14,0001403161,V,VISA INC.
51,0001067839,QQQ,"INVESCO QQQ TRUST, SERIES 1"


## 2. Structured Core

Start with macro series from FRED and optionally enrich ticker metadata with FMP if an API key is available.


In [63]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')
fred_series = {
    'FEDFUNDS': 'Fed Funds Rate',
    'CPIAUCSL': 'CPI',
    'UNRATE': 'Unemployment Rate',
    'DGS10': '10Y Treasury',
    'DGS2': '2Y Treasury',
}
fred_frames = []
for series_id, title in fred_series.items():
    fred_url = (
        'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    )
    payload = fetch_json(fred_url)
    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]
    frame['series_id'] = series_id
    frame['title'] = title
    fred_frames.append(frame)
fred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()
fred_df.to_csv(EXTERNAL_DIR / 'fred' / 'macro_series.csv', index=False)
fred_df.tail(10)

,date,value,series_id,title
32517,2026-03-27,3.88,DGS2,2Y Treasury
32518,2026-03-30,3.82,DGS2,2Y Treasury
32519,2026-03-31,3.79,DGS2,2Y Treasury
32520,2026-04-01,3.81,DGS2,2Y Treasury
32521,2026-04-02,3.79,DGS2,2Y Treasury
32522,2026-04-03,3.84,DGS2,2Y Treasury
32523,2026-04-06,3.84,DGS2,2Y Treasury
32524,2026-04-07,3.81,DGS2,2Y Treasury
32525,2026-04-08,3.79,DGS2,2Y Treasury
32526,2026-04-09,3.78,DGS2,2Y Treasury


In [58]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')
if FMP_API_KEY and not company_master.empty:
    sample_tickers = company_master['ticker'].head(5).tolist()
    profile_frames = []
    for symbol in sample_tickers:
        fmp_url = f'https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={FMP_API_KEY}'
        profile_payload = fetch_json(fmp_url)
        profile_frames.append(pd.DataFrame(profile_payload if isinstance(profile_payload, list) else [profile_payload]))
    fmp_profiles = pd.concat(profile_frames, ignore_index=True) if profile_frames else pd.DataFrame()
    fmp_profiles.to_csv(EXTERNAL_DIR / 'fmp' / 'company_profiles_preview.csv', index=False)
    display_columns = [col for col in ['symbol', 'companyName', 'sector', 'industry', 'marketCap'] if col in fmp_profiles.columns]
    display(fmp_profiles[display_columns].head())
else:
    print('FMP_API_KEY not set. Skip this cell for now or add a free-tier key to test profile enrichment.')

,symbol,companyName,sector,industry,marketCap
0,NVDA,NVIDIA Corporation,Technology,Semiconductors,4584652476141
1,AAPL,Apple Inc.,Technology,Consumer Electronics,3828515423772
2,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,3837652369144
3,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,2753943398100
4,AMZN,"Amazon.com, Inc.",Consumer Cyclical,Specialty Retail,2558990436991


In [68]:
fmp_profiles.head() 

,symbol,price,marketCap,beta,lastDividend,range,change,changePercentage,volume,averageVolume,...,city,state,zip,image,ipoDate,defaultImage,isEtf,isActivelyTrading,isAdr,isFund
0,NVDA,188.63,4584652476141,2.335,0.04,95.04-212.19,4.72,2.566470,159395820,180352446,...,Santa Clara,CA,95051,https://images.financialmodelingprep.com/symbo...,1999-01-22,False,False,True,False,False
1,AAPL,260.48,3828515423772,1.109,1.04,189.81-288.62,-0.01,-0.003839,23433118,47424558,...,Cupertino,CA,95014,https://images.financialmodelingprep.com/symbo...,1980-12-12,False,False,True,False,False
2,GOOGL,317.24,3837652369144,1.128,0.84,146.1-349,-1.25,-0.392480,18914274,33724669,...,Mountain View,CA,94043,https://images.financialmodelingprep.com/symbo...,2004-08-19,False,False,True,False,False
3,MSFT,370.87,2753943398100,1.107,3.48,355.67-555.45,-2.20,-0.589700,27933678,36853329,...,Redmond,WA,98052-6399,https://images.financialmodelingprep.com/symbo...,1986-03-13,False,False,True,False,False
4,AMZN,238.38,2558990436991,1.383,0.00,165.29-258.6,4.73,2.024400,56861543,50336243,...,Seattle,WA,98109-5210,https://images.financialmodelingprep.com/symbo...,1997-05-15,False,False,True,False,False


## 3. SEC Filing Metadata + Company Facts

Pull official filing metadata and company facts for one sample ticker so you can validate the SEC ingestion path before scaling it.


In [69]:
sample_ticker = company_master['ticker'].iloc[0] if not company_master.empty else 'AAPL'
sample_cik = company_master.loc[company_master['ticker'].eq(sample_ticker), 'cik'].iloc[0] if not company_master.empty else '0000320193'
submissions_url = f'https://data.sec.gov/submissions/CIK{sample_cik}.json'
submissions = fetch_json(submissions_url, headers=SEC_HEADERS)
recent_all = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))
recent_all = recent_all[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].copy()
recent = recent_all.head(20).copy()
recent.to_csv(EXTERNAL_DIR / 'sec' / f'{sample_ticker.lower()}_filings_preview.csv', index=False)
display(recent.head(10))
facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{sample_cik}.json'
company_facts = fetch_json(facts_url, headers=SEC_HEADERS)
us_gaap = company_facts.get('facts', {}).get('us-gaap', {})
fact_names = list(us_gaap.keys())[:15]
print('***********')
pd.DataFrame({'ticker': [sample_ticker] * len(fact_names), 'fact_tag': fact_names})

,accessionNumber,filingDate,form,primaryDocument
0,0000102909-26-000426,2026-03-26,SCHEDULE 13G/A,xslSCHEDULE_13G_X02/primary_doc.xml
1,0001199039-26-000003,2026-03-24,4,xslF345X06/wk-form4_1774386816.xml
2,0001725292-26-000002,2026-03-20,4,xslF345X06/wk-form4_1774052040.xml
3,0001526111-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051982.xml
4,0001696841-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051862.xml
5,0001283854-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051812.xml
6,0001347842-26-000011,2026-03-20,4,xslF345X06/wk-form4_1774051696.xml
7,0001588670-26-000010,2026-03-20,4,xslF345X06/wk-form4_1774051632.xml
8,0001197649-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051558.xml
9,0001628280-26-020268,2026-03-20,144,xsl144X01/primary_doc.xml


***********


,ticker,fact_tag
0,NVDA,AcceleratedShareRepurchaseProgramAdjustment
1,NVDA,AcceleratedShareRepurchasesFinalPricePaidPerShare
2,NVDA,AcceleratedShareRepurchasesSettlementPaymentOr...
3,NVDA,AccountsPayableCurrent
4,NVDA,AccountsReceivableNetCurrent
5,NVDA,AccruedIncomeTaxesNoncurrent
6,NVDA,AccruedIncomeTaxesPayable
7,NVDA,AccruedLiabilitiesCurrent
8,NVDA,AccruedProfessionalFeesCurrentAndNoncurrent
9,NVDA,AccruedRentCurrent


## 4. Text Pipeline / Vector Preview

Take one recent 10-K or 10-Q, fetch the filing text, clean it, and create vector-ready chunks with metadata. This is a preview of what will later go into the vector database.


In [70]:
candidate_forms = recent_all[recent_all['form'].isin(['10-K', '10-Q'])].copy()
if candidate_forms.empty:
    raise ValueError(f'No recent 10-K/10-Q found for {sample_ticker}')
filing_row = candidate_forms.iloc[0]
accession = filing_row['accessionNumber'].replace('-', '')
primary_doc = filing_row['primaryDocument']
filing_url = f'https://www.sec.gov/Archives/edgar/data/{int(sample_cik)}/{accession}/{primary_doc}'
filing_html = fetch_text(filing_url, headers=SEC_HEADERS)
filing_text = html_to_text(filing_html)
chunks = simple_chunk(filing_text, chunk_size=1800)
chunk_preview = pd.DataFrame({
    'ticker': sample_ticker,
    'cik': sample_cik,
    'form_type': filing_row['form'],
    'filing_date': filing_row['filingDate'],
    'chunk_id': [f'{sample_ticker}-{i:03d}' for i in range(len(chunks[:5]))],
    'text_preview': chunks[:5],
})
chunk_preview.to_csv(INTERIM_DIR / 'vector_preview' / f'{sample_ticker.lower()}_sec_chunks_preview.csv', index=False)
chunk_preview

,ticker,cik,form_type,filing_date,chunk_id,text_preview
0,NVDA,0001045810,10-K,2026-02-25,NVDA-000,nvda-20260125 0001045810 2026 FY false 362 460...
1,NVDA,0001045810,10-K,2026-02-25,NVDA-001,01-26 0001045810 us-gaap:AccumulatedOtherCompr...
2,NVDA,0001045810,10-K,2026-02-25,NVDA-002,sesMember 2024-01-29 2025-01-26 0001045810 us-...
3,NVDA,0001045810,10-K,2026-02-25,NVDA-003,atentsAndLicensedTechnologyMember 2025-01-26 0...
4,NVDA,0001045810,10-K,2026-02-25,NVDA-004,currentAssetsMember us-gaap:FairValueInputsLev...


## 5. News / Narrative + Derived Feature Preview

Use GDELT for a quick free narrative pass, then create a tiny derived feature snapshot so you can sanity-check how the structured and text layers will come together later.


In [61]:
query = quote(sample_ticker)
gdelt_url = (
    'https://api.gdeltproject.org/api/v2/doc/doc'
    f'?query={query}&mode=artlist&maxrecords=10&format=json'
)
gdelt_payload = fetch_json(gdelt_url)
articles = pd.DataFrame(gdelt_payload.get('articles', []))
articles.to_csv(EXTERNAL_DIR / 'gdelt' / f'{sample_ticker.lower()}_news_preview.csv', index=False)
display(articles[['title', 'domain', 'seendate', 'url']].head(10) if not articles.empty else pd.DataFrame())
derived_snapshot = pd.DataFrame([{
    'ticker': sample_ticker,
    'recent_filings_count': len(recent),
    'vector_chunk_count_preview': len(chunks),
    'news_article_count_preview': len(articles),
    'macro_series_tracked': len(fred_series),
}])
derived_snapshot.to_csv(INTERIM_DIR / 'derived_feature_preview.csv', index=False)
derived_snapshot

,title,domain,seendate,url
0,FinancialContent - YieldMax NVDA Option Income...,markets.financialcontent.com,20260309T184500Z,https://markets.financialcontent.com/stocks/qu...
1,How NVIDIA Corporation ( NVDA ) Is Shaping the...,insidermonkey.com,20260323T174500Z,https://www.insidermonkey.com/blog/how-nvidia-...
2,New Street Research Adds NVIDIA ( NVDA ) to Be...,finance.yahoo.com,20260322T110000Z,https://finance.yahoo.com/markets/stocks/artic...
3,Where is NVIDIA Corporation ( NVDA ) Headed Ac...,insidermonkey.com,20260119T113000Z,https://www.insidermonkey.com/blog/where-is-nv...
4,Raymond James Raises PT on NVIDIA ( NVDA ) Stock,insidermonkey.com,20260323T213000Z,https://www.insidermonkey.com/blog/raymond-jam...
5,Tigress Financial Raises NVIDIA ( NVDA ) Price...,insidermonkey.com,20260324T131500Z,https://www.insidermonkey.com/blog/tigress-fin...
6,Where is NVIDIA Corporation ( NVDA ) Headed Ac...,finance.yahoo.com,20260119T101500Z,https://finance.yahoo.com/news/where-nvidia-co...
7,Here How Much You Would Have Made Owning NVIDI...,benzinga.com,20260325T000000Z,https://www.benzinga.com/insights/news/26/03/5...
8,Nvidia ( NVDA ) Deal With Competitor Marvell (...,insidermonkey.com,20260407T154500Z,https://www.insidermonkey.com/blog/nvidias-nvd...
9,What Does the Street Think About NVIDIA Corpor...,insidermonkey.com,20260208T100000Z,https://www.insidermonkey.com/blog/what-does-t...


,ticker,recent_filings_count,vector_chunk_count_preview,news_article_count_preview,macro_series_tracked
0,NVDA,20,208,10,5
